# Build Canonical FPA Splits (`train_val` / `val_val` / `holdout_test`)

This notebook creates the fixed split files used by experiment runners directly under `data/fpa/`:

- `campaigns_fpa_train_val.csv`
- `stats_fpa_train_val.csv`
- `campaigns_fpa_val_val.csv`
- `stats_fpa_val_val.csv`
- `campaigns_fpa_holdout_test.csv`
- `stats_fpa_holdout_test.csv`

Source files:

- `campaigns_fpa_filtered_train_final.csv`
- `stats_fpa_filtered_train_final.csv`
- `campaigns_fpa_filtered_test_final.csv`
- `stats_fpa_filtered_test_final.csv`


In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# Parameters
VAL_FRACTION = 0.2
SPLIT_SEED = 42
OVERWRITE = True

# Resolve project root robustly whether notebook runs from repo root or example_notebooks/.
cwd = Path.cwd().resolve()
if (cwd / "config.py").exists():
    project_root = cwd
elif (cwd.parent / "config.py").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Cannot locate project root (config.py not found in cwd or parent).")

data_fpa_dir = project_root / "data" / "fpa"

src_campaigns_train = data_fpa_dir / "campaigns_fpa_filtered_train_final.csv"
src_stats_train = data_fpa_dir / "stats_fpa_filtered_train_final.csv"
src_campaigns_test = data_fpa_dir / "campaigns_fpa_filtered_test_final.csv"
src_stats_test = data_fpa_dir / "stats_fpa_filtered_test_final.csv"

dst_campaigns_train_val = data_fpa_dir / "campaigns_fpa_train_val.csv"
dst_stats_train_val = data_fpa_dir / "stats_fpa_train_val.csv"
dst_campaigns_val_val = data_fpa_dir / "campaigns_fpa_val_val.csv"
dst_stats_val_val = data_fpa_dir / "stats_fpa_val_val.csv"
dst_campaigns_holdout = data_fpa_dir / "campaigns_fpa_holdout_test.csv"
dst_stats_holdout = data_fpa_dir / "stats_fpa_holdout_test.csv"

for src_path in [src_campaigns_train, src_stats_train, src_campaigns_test, src_stats_test]:
    if not src_path.exists():
        raise FileNotFoundError(f"Missing source file: {src_path}")

for dst_path in [
    dst_campaigns_train_val,
    dst_stats_train_val,
    dst_campaigns_val_val,
    dst_stats_val_val,
    dst_campaigns_holdout,
    dst_stats_holdout,
]:
    if dst_path.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite existing file: {dst_path}")

campaigns_train = pd.read_csv(src_campaigns_train)
stats_train = pd.read_csv(src_stats_train)
campaigns_test = pd.read_csv(src_campaigns_test)
stats_test = pd.read_csv(src_stats_test)

if "campaign_id" not in campaigns_train.columns or "campaign_id" not in stats_train.columns:
    raise KeyError("Expected 'campaign_id' column in train source files.")

n_val = max(1, int(round(len(campaigns_train) * VAL_FRACTION)))
val_campaigns = (
    campaigns_train
    .sample(n=n_val, random_state=SPLIT_SEED)
    .sort_values("campaign_id")
    .reset_index(drop=True)
)
val_campaign_ids = set(val_campaigns["campaign_id"].astype(int).tolist())

train_val_campaigns = (
    campaigns_train[~campaigns_train["campaign_id"].astype(int).isin(val_campaign_ids)]
    .sort_values("campaign_id")
    .reset_index(drop=True)
)

train_val_campaign_ids = set(train_val_campaigns["campaign_id"].astype(int).tolist())
train_val_stats = stats_train[stats_train["campaign_id"].astype(int).isin(train_val_campaign_ids)].copy()
val_val_stats = stats_train[stats_train["campaign_id"].astype(int).isin(val_campaign_ids)].copy()

# holdout_test is copied from existing filtered test split.
holdout_campaigns = campaigns_test.copy()
holdout_stats = stats_test.copy()

train_val_campaigns.to_csv(dst_campaigns_train_val, index=False)
train_val_stats.to_csv(dst_stats_train_val, index=False)
val_campaigns.to_csv(dst_campaigns_val_val, index=False)
val_val_stats.to_csv(dst_stats_val_val, index=False)
holdout_campaigns.to_csv(dst_campaigns_holdout, index=False)
holdout_stats.to_csv(dst_stats_holdout, index=False)

summary = pd.DataFrame([
    {"split": "train_val", "campaigns": len(train_val_campaigns), "stats_rows": len(train_val_stats)},
    {"split": "val_val", "campaigns": len(val_campaigns), "stats_rows": len(val_val_stats)},
    {"split": "holdout_test", "campaigns": len(holdout_campaigns), "stats_rows": len(holdout_stats)},
])

print("Created canonical split files in:", data_fpa_dir)
print("VAL_FRACTION=", VAL_FRACTION, "SPLIT_SEED=", SPLIT_SEED)
summary

Created canonical split files in: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa
VAL_FRACTION= 0.2 SPLIT_SEED= 42


,split,campaigns,stats_rows
0,train_val,1027,1246481
1,val_val,257,286690
2,holdout_test,1285,1503083


In [3]:
pd.DataFrame({
    "file": [
        str(dst_campaigns_train_val),
        str(dst_stats_train_val),
        str(dst_campaigns_val_val),
        str(dst_stats_val_val),
        str(dst_campaigns_holdout),
        str(dst_stats_holdout),
    ],
    "exists": [
        dst_campaigns_train_val.exists(),
        dst_stats_train_val.exists(),
        dst_campaigns_val_val.exists(),
        dst_stats_val_val.exists(),
        dst_campaigns_holdout.exists(),
        dst_stats_holdout.exists(),
    ],
})

,file,exists
0,/Users/amsafin/code/local_ml/rl/bat-autobiddin...,True
1,/Users/amsafin/code/local_ml/rl/bat-autobiddin...,True
2,/Users/amsafin/code/local_ml/rl/bat-autobiddin...,True
3,/Users/amsafin/code/local_ml/rl/bat-autobiddin...,True
4,/Users/amsafin/code/local_ml/rl/bat-autobiddin...,True
5,/Users/amsafin/code/local_ml/rl/bat-autobiddin...,True
